# Transfer Bayesian Optimization：過去の知識を新しい最適化へ活用する

このNotebookでは、**過去の実験（source task）から得た知識を新しい実験（target task）の候補提案に活用する**転移ベイズ最適化を学びます。

1. Warm-startで過去の良好条件を初期候補にする
2. TAFで過去GPの予測を獲得関数へ取り入れる
3. RGPEで過去GPの有用性を順位情報から推定する
4. q-batchのSequential/Joint方式を比較する
5. Mixed変数の合法値を確認する

**重要：** ソースの測定値をターゲットの測定値として使い回しません。ソースGPは学習済みモデルを渡し、ターゲットGPは既存の候補提案関数が内部で学習します。

詳しい数式は[理論ガイド](../../docs/theory/26_transfer_bayesian_optimization.md)、APIは[利用ガイド](../../docs/guide/26_transfer_bayesian_optimization.md)を参照してください。


## 0. 転移ベイズ最適化の全体像

| 方法 | 過去タスクを使う場所 | 重みの扱い |
|---|---|---|
| Warm-start | 新しい実験の**初期候補**を選ぶ | GPのアンサンブル重みなし |
| TAF | **獲得関数**にソースGPの情報を追加 | 利用者が指定 |
| RGPE | **ソース・ターゲットGPの予測**を統合 | 順位の一致度から推定 |

通常のBOはターゲット観測のみを使います。転移BOは、少数のターゲット観測でも過去の知識を活かせる可能性がありますが、タスクが似ていないと**負の転移**が生じることがあります。

In [ ]:
import torch
from robotorchan.models import SingleTaskGP
from robotorchan.optim.transfer.taf_sequential import propose_taf_candidate
from robotorchan.optim.transfer.rgpe_sequential import propose_rgpe_candidate

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])
def target_objective(x):
    return -((x - 0.75) ** 2)

source_x = torch.linspace(0, 1, 6).unsqueeze(-1)
source_y = -((source_x - 0.7) ** 2)
source_model = SingleTaskGP(source_x, source_y)
source_model.fit()
source_model.eval()
target_x = torch.tensor([[0.0], [0.3], [1.0]])
target_y = target_objective(target_x)
# source_model は過去タスクのGP、target_x / target_y は新しいタスクの観測です。
print('source_x:', source_x.shape, 'source_y:', source_y.shape)
print('target_x:', target_x.shape, 'target_y:', target_y.shape)


### ソースとターゲットの目的関数を可視化する

合成例ではソースの最適位置は0.7、ターゲットは0.75です。両者は近いですが同じではありません。

図では真の関数を表示しますが、**実際の転移BOでは未知**であり、ソースGPにはソース観測、ターゲット側にはターゲット観測だけを与えます。

In [ ]:
import matplotlib.pyplot as plt

plot_x = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_x.squeeze(-1), -((plot_x-0.7)**2).squeeze(-1), label="Source objective (demo only)")
ax.plot(plot_x.squeeze(-1), target_objective(plot_x).squeeze(-1), label="Target objective (demo only)")
ax.scatter(source_x.squeeze(-1), source_y.squeeze(-1), marker="x", label="Source observations")
ax.scatter(target_x.squeeze(-1), target_y.squeeze(-1), label="Target observations")
ax.set(xlabel="x", ylabel="Response", title="Related but different tasks")
ax.legend()
plt.show()

## 1. Warm-start：過去の良好な条件から始める

まず前のセルで作った変数を確認します。

- `bounds`：探索範囲。形状は `(2, d)` で、1行目が下限、2行目が上限です。今回は `d=1` なので0〜1の範囲です。
- `source_x` / `source_y`：過去タスクの入力と評価値。今回の例では0.7付近が良好です。
- `target_x` / `target_y`：新しいタスクで**実際に評価済み**の入力と評価値。こちらは0.75付近が良好です。
- `source_model`：ソース観測から構築したGPモデル。ここでは説明用に単純な1次元GPを使います。

`select_warm_start_points` は過去の良好な入力を参考に、ターゲットで**これから評価する位置**を選びます。返り値 `initial_x` は測定結果ではありません。

次のセルでは教育用の人工目的関数 `target_objective` を直接呼んで評価します。実務ではこの部分を実験装置やシミュレーションの実行に置き換えてください。なお、このセルはWarm-startの仕組みを示す独立例であり、後続のTAFに使う `target_x` を自動更新しません。

In [ ]:
from robotorchan.optim.transfer.warm_start import select_warm_start_points

initial_x = select_warm_start_points(
    bounds, 3, source_x=source_x, source_y=source_y, seed=0,
)
initial_y = target_objective(initial_x)
print('Warm-start target observations:', initial_x, initial_y)

### Warm-start候補と実測値を区別する

`select_warm_start_points()` が返す `initial_x` は過去データから選んだ**候補の座標**です。`target_objective(initial_x)` を実行して初めてターゲットでの観測になります。

ここではWarm-startの挙動を独立して確認するため、後続のTAF用 `target_x/target_y` には自動で追加しません。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_x.squeeze(-1), target_objective(plot_x).squeeze(-1), label="Target objective (demo only)")
ax.scatter(target_x.squeeze(-1), target_y.squeeze(-1), label="Existing target observations")
ax.scatter(initial_x.squeeze(-1), initial_y.squeeze(-1), marker="*", s=140, label="Warm-start evaluations")
ax.set(xlabel="x", ylabel="Response", title="Warm-start evaluated on target task")
ax.legend()
plt.show()

## 2. TAF：ソース情報を獲得関数に加える

通常のベイズ最適化では、現在のターゲットGPの**予測平均と不確実性**から、次に試す点を決めます。TAF（Transfer Acquisition Function）はそこへ、過去タスクの予測平均から得られる「良さそうな領域」の情報を加えます。

最大化では概念的に次のように表します。

\[
a_{\mathrm{TAF}}(x)=w_t\operatorname{EI}_t(x)+\sum_iw_i\max(\mu_i(x)-f_t^*,0).
\]

ここで `EI` はターゲットの期待改善量、`μ_i` はソースモデルの予測平均、`f_t^*` はターゲットの最良観測値です。

**重みの順番に注意してください。** `weights = [0.5, 0.5]` は「ソースモデルの重み0.5、ターゲットモデルの重み0.5」です。ソースが2個なら `[source1, source2, target]` の順になります。重みは非負で、合計を1にします。

次のループでは、**提案 → 人工目的関数で評価 → 観測を追加**を2回繰り返します。毎回ターゲットの観測データが増える点に注目してください。ソースとターゲットが似ていないとき、ソース重みが大き過ぎると「負の転移」が起こり得ます。

In [ ]:
weights = torch.tensor([0.5, 0.5])
for iteration in range(2):
    candidate = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        raw_samples=16, num_restarts=2, seed=iteration,
    )
    observation = target_objective(candidate)
    target_x = torch.cat([target_x, candidate])
    target_y = torch.cat([target_y, observation])
    print(f'TAF iteration={iteration + 1}, x={candidate.item():.4f}, best={target_y.max().item():.4f}')

### TAFでの最良観測値の変化を確認する

TAFではターゲットの新しい観測値を追加してから再び候補を提案します。**ソースGPは過去タスクの学習済みモデルのまま**で、ターゲットGPは候補提案関数内で毎回学習されます。

`weights=[source,target]` は固定です。ソースとターゲットが似ていない場合は、重みの見直しが必要になることがあります。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_x.squeeze(-1), target_objective(plot_x).squeeze(-1), label="Target objective (demo only)")
ax.scatter(target_x.squeeze(-1), target_y.squeeze(-1), label="Target observations after TAF")
ax.axhline(target_y.max().item(), linestyle="--", color="gray", label="Best observed target value")
ax.set(xlabel="x", ylabel="Response", title="TAF observation history")
ax.legend()
plt.show()

## 3. RGPE：どの過去モデルを信頼するか学習する

RGPE（Ranking-weighted Gaussian Process Ensemble）は、ソースモデルとターゲットモデルの**候補の良し悪しの順位**を比較して重みを決めます。

1. ソースモデルで、現在のターゲット観測点を予測する
2. ターゲットモデルについては、対象点を学習から除いた **LOO（Leave-One-Out）予測**を使う
3. 実測順位と予測順位の不一致を数える
4. 観測点のペアをブートストラップ抽出し、順位の良いモデルに重みを与える
5. 重み付きのモデル予測を統合し、次の候補を提案する

LOOを使う理由は、ターゲットモデルが**学習に使用した点をそのまま予測すると有利になり過ぎる**ためです。計算には複数回のGP学習が必要です。

次のセルの戻り値は `(candidate, learned_weights)` です。TAFと違い、RGPEでは重みを自分で指定しません。重みは **ソースモデル、ターゲットモデル** の順で表示されます。

`n_bootstrap=8` は説明用の軽量設定です。実務では推定の安定性を確認し、必要に応じて回数を増やしてください。

In [ ]:
candidate, learned_weights = propose_rgpe_candidate(
    [source_model], target_x, target_y, bounds,
    n_bootstrap=8, raw_samples=16, num_restarts=2, seed=0,
)
print('RGPE candidate:', candidate)
print('Source / target weights:', learned_weights)

### RGPEが学習した重みを可視化する

RGPEの重みは `[source model, target model]` の順です。**重みが大きいことは、そのモデルが今回のターゲットの順位関係を比較的よく説明したこと**を意味します。重みは観測が増えると変化し得ます。

`n_bootstrap=8` は高速な説明用設定であり、安定した推定を保証するものではありません。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Source GP", "Target GP"], learned_weights.detach().cpu().reshape(-1).tolist())
ax.set(ylabel="RGPE weight", title="Learned ranking-based weights")
plt.show()

## 4. 結果の読み方と比較実験

これまでの例はAPIの使い方を理解するためのものです。**1回の候補提案だけでTAFとRGPEの優劣は判断できません。**

公平な比較では、同じターゲット初期観測、同じ評価予算、複数の乱数シードを用意します。横軸をターゲットでの実評価回数、縦軸をそれまでの最良観測値（best-so-far）として比較します。通常BOを基準として加え、ソースとターゲットが似ている場合だけでなく、異なる場合も試してください。

比較用のスクリプトは `examples/benchmarks/transfer_bo.py` です。RGPEはLOO再学習の分だけ計算時間が増える可能性があります。

### TAFとRGPEの違いを振り返る

TAFは**ソースモデルに与える重みを利用者が設定**し、RGPEは**ターゲット観測の順位情報を使って重みを推定**します。

両者とも候補を返すだけで実験は実行しません。RGPEの候補も実際に測定する場合は、`target_objective(candidate)` で評価し、ターゲット観測へ追加してください。

## 5. q>1：複数の候補をまとめて提案する

`q` は**1回の呼び出しで提案する候補の個数**です。例えば `q=2` は2つの実験条件を提案します。提案しただけでは測定は完了していません。

- **Sequential greedy**：1点選び、その点でのターゲットGP予測平均を**仮の観測値（fantasy）**として条件付けて次の点を選びます。実際に測定したわけではありません。
- **Joint batch**：2点の組をまとめて最適化します。ターゲットのqEIは2点の予測の相関も考慮します。

次のセルではTAFの2方式を同じデータから比較します。 `batch_strategy="sequential"` または `"joint"` を指定します。 `mc_samples` はJointでのMonte Carloサンプル数です。

出力の形状はどちらも `(2, 1)`（2候補×1変数）です。出力値は候補の**座標**であり、目的関数の評価値ではありません。

In [ ]:
for strategy in ("sequential", "joint"):
    batch = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "TAF candidates:", batch)


### SequentialとJointのq-batchの違い

`q=2` なら2つの候補を提案します。Sequentialは1点目を選び、GP予測平均を**仮の観測（fantasy）**として次を選びます。Jointは2点の組をまとめて評価します。

**Fantasyは実測値ではありません。** 提案後の2点を測定し終わってからターゲット観測へ追加します。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.scatter(target_x.squeeze(-1), torch.zeros_like(target_x.squeeze(-1)), marker="x", label="Observed target X")
for idx, strategy in enumerate(("sequential", "joint")):
    batch = propose_taf_candidate(
        [source_model], target_x, target_y, bounds, weights,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    ax.scatter(batch.squeeze(-1), torch.full_like(batch.squeeze(-1), idx+1, dtype=torch.double), label=f"{strategy} proposals")
ax.set(xlim=(0, 1), ylim=(-0.5, 2.5), yticks=[0, 1, 2], yticklabels=["Observed", "Sequential", "Joint"], xlabel="x", title="q=2 candidate locations")
ax.legend()
plt.show()

## 6. Mixed変数：連続値とカテゴリを同時に扱う

製造条件では、例えば「温度は連続値」「材料の種類はカテゴリ」のように、変数の種類が混在します。ここでは次の2変数を考えます。

| 列番号 | 意味 | 値 |
| --- | --- | --- |
| 0 | 連続変数（例：温度を0〜1に正規化） | 0〜1 |
| 1 | カテゴリ（例：材料A/B） | 0または1 |

`mixed_x` の形状は `(4, 2)`、`mixed_y` は `(4, 1)` です。`cat_dims=[1]` は「**1列目ではなく、0始まりで数えた列番号1（2列目）**がカテゴリ」を意味します。

`fixed_features_list=[{1: 0.0}, {1: 1.0}]` は、カテゴリ列に許される値を列挙しています。0と1はカテゴリの識別子であり、**0.5という中間カテゴリは存在しません**。

この例は転移先だけの振る舞いを理解するため、ソースモデルを空リスト `[]` にしています。そのためTAFの重みはターゲット1個分の `torch.ones(1)` です。これは**Mixed最適化APIの動作確認例**であり、ソース知識を活用した転移性能の実証ではありません。RGPEでもソースが空ならターゲットだけを利用します。

`q=2` としたときの出力は `(2, 2)`（2候補×2変数）です。2列目が0か1になっているかを確認してください。Joint方式は候補の組を最適化するため、Sequentialと異なる結果になることがあります。

### Mixed変数のカテゴリを確認する

ここでは `x0` が連続変数、`x1` がカテゴリ値 `{0,1}` です。カテゴリの数値は識別子であり、`0.5` のような中間カテゴリは存在しません。

`fixed_features_list` は候補生成で合法なカテゴリを列挙します。`q=2` の戻り値は `[2,2]` で、**候補数と変数数がたまたま同じ**です。

### Mixedでソースモデルが空の例の位置付け

このセルは**Mixed候補生成APIの使い方を確認する独立例**です。ソースモデルを `[]` としているため、過去タスクからの転移効果を示しているわけではありません。

転移効果を検証するには、Mixed変数を含むソースデータでGPを学習し、そのソースGPを渡して比較する必要があります。

In [ ]:
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_x = torch.tensor([[0.0, 0.0], [0.5, 1.0], [1.0, 0.0], [0.2, 1.0]])
def mixed_objective(x):
    return -((x[:, :1] - 0.7) ** 2) + 0.1 * x[:, 1:2]

mixed_y = mixed_objective(mixed_x)
categories = [{1: 0.0}, {1: 1.0}]
for strategy in ("sequential", "joint"):
    taf_batch = propose_taf_candidate(
        [], mixed_x, mixed_y, mixed_bounds, torch.ones(1),
        cat_dims=[1], fixed_features_list=categories,
        q=2, batch_strategy=strategy, mc_samples=32,
        raw_samples=16, num_restarts=2, seed=0,
    )
    print(strategy, "Mixed TAF:", taf_batch)

rgpe_batch, rgpe_weights = propose_rgpe_candidate(
    [], mixed_x, mixed_y, mixed_bounds,
    cat_dims=[1], fixed_features_list=categories,
    q=2, batch_strategy="joint", n_bootstrap=8,
    mc_samples=32, raw_samples=16, num_restarts=2, seed=0,
)
print("Mixed RGPE joint:", rgpe_batch, rgpe_weights)

print('Mixed input shape:', mixed_x.shape)
print('Allowed categories:', categories)


## 7. 実装上の責務と学習のまとめ

| 目的 | 既存API | 注意点 |
|---|---|---|
| ソースGPの学習 | `robotorchan.models.SingleTaskGP.fit()` | **学習済みモデル**を転移関数へ渡す |
| 過去の良好条件から初期候補 | `select_warm_start_points()` | 候補はターゲットで未評価 |
| 固定重みの転移獲得関数 | `propose_taf_candidate()` | ソースとターゲットの重み順に注意 |
| 順位による重み推定 | `propose_rgpe_candidate()` | LOOとbootstrapを使用 |
| 複数候補 | `q`, `batch_strategy` | fantasyと実測値を混同しない |
| Mixedカテゴリ | `cat_dims`, `fixed_features_list` | 合法カテゴリを明示 |

**重要：** TAF・RGPEの提案関数は内部でターゲットGPを学習します。ソースの測定値をターゲットの観測値として登録せず、実際のターゲット実験の結果だけを追加してください。

本Notebookは使い方を示す小規模例です。性能比較には通常BO、複数seed、同じターゲット評価予算、似ていないソースを含む条件が必要です。